<a href="https://colab.research.google.com/github/nabanita-data/data-portfolio/blob/main/HRIS_Data_Validation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# HRIS Data Validation Project
This notebook demonstrates how I built a simple HR data validation pipeline in Python.  
It includes synthetic employee and department datasets, common data quality checks, and an automated validation report.

In [6]:
!pip install faker

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 37.3 MB/s eta 0:00:00


## 1. Generate synthetic HR dataset  
This section creates a synthetic HR dataset using Faker. It includes 100 employees and 5 departments.  
The dataset is safe to share and realistic enough to simulate HRIS data validation scenarios.  

In [7]:
# Generating a synthetic HR dataset
import pandas as pd
import numpy as np
from faker import Faker
import random
from datetime import datetime, timedelta

fake = Faker()
random.seed(42)

# departments
departments = [
    {"department_id": 1, "department_name": "HR"},
    {"department_id": 2, "department_name": "Engineering"},
    {"department_id": 3, "department_name": "Finance"},
    {"department_id": 4, "department_name": "Operations"},
    {"department_id": 5, "department_name": "Sales"},
]
pd.DataFrame(departments).to_csv("sample_departments.csv", index=False)

# employees
rows = []
for i in range(100):
    emp_id = 1000 + i
    first = fake.first_name()
    last = fake.last_name()
    email = f"{first.lower()}.{last.lower()}@example.com"
    hire_date = fake.date_between(start_date='-5y', end_date='today')
    term_date = "" if random.random() > 0.2 else hire_date + timedelta(days=random.randint(30, 500))
    salary = round(random.uniform(25000, 100000), 2)
    dept = random.choice(departments)["department_id"]
    rows.append({
        "employee_id": emp_id,
        "first_name": first,
        "last_name": last,
        "email": email,
        "hire_date": hire_date,
        "termination_date": term_date,
        "department_id": dept,
        "salary": salary
    })

df = pd.DataFrame(rows)
df.to_csv("sample_employees.csv", index=False)
print("Created CSVs:", "sample_employees.csv and sample_departments.csv")

Created CSVs: sample_employees.csv and sample_departments.csv


## 2. Run validation checks  
This section applies data quality checks on the generated HR dataset:  
- Missing or duplicate IDs  
- Invalid email formats  
- Future or malformed dates  
- Termination dates before hire  
- Salary range violations  
- Invalid department references  

In [10]:
import pandas as pd
import json, os
from datetime import datetime

BASE = os.getcwd()
EMP = "sample_employees.csv"
DEPT = "sample_departments.csv"

def read(path):
    return pd.read_csv(os.path.join(BASE, path))

def check_not_null(df, col):
    nulls = int(df[col].isnull().sum())
    return {"check": f"not_null:{col}", "passed": nulls==0, "null_count": nulls}

def check_unique(df, col):
    dup = int(df[col].duplicated().sum())
    return {"check": f"unique:{col}", "passed": dup==0, "dup_count": dup}

def check_email_simple(df, col):
    s = df[col].astype(str)
    def valid_email(x):
        if '@' not in x:
            return False
        local, domain = x.split('@', 1)
        return '.' in domain and local != '' and domain.split('.')[-1].isalpha()
    mask = s.apply(valid_email)
    fail = int((~mask).sum())
    return {"check": f"email_format:{col}", "passed": fail==0, "fail_count": fail}

def check_date_past(df, col):
    parsed = pd.to_datetime(df[col], errors='coerce')
    today = pd.Timestamp(datetime.today().date())
    future_count = int((parsed > today).sum())
    parse_errors = int(parsed.isna().sum())
    return {"check": f"date_past:{col}", "passed": future_count==0 and parse_errors==0,
            "future_count": future_count, "parse_errors": parse_errors}

def check_term_after_hire(df, col_hire, col_term):
    hire = pd.to_datetime(df[col_hire], errors='coerce')
    term = pd.to_datetime(df[col_term], errors='coerce')
    mask = term.notna() & hire.notna()
    bad = int((term[mask] < hire[mask]).sum())
    return {"check": "term_after_hire", "passed": bad==0, "bad_count": bad}

def check_numeric_range(df, col, minimum, maximum):
    num = pd.to_numeric(df[col], errors='coerce')
    non_numeric = int(num.isna().sum())
    below = int((num < minimum).sum())
    above = int((num > maximum).sum())
    return {"check": f"numeric_range:{col}",
            "passed": below==0 and above==0 and non_numeric==0,
            "below": below, "above": above, "non_numeric": non_numeric}

def check_fk(df, col, ref_df, ref_col):
    miss = ~df[col].isin(ref_df[ref_col])
    miss_count = int(miss.sum())
    return {"check": f"foreign_key:{col}", "passed": miss_count==0, "missing_count": miss_count}

def run(base):
    emp = read(EMP)
    dept = read(DEPT)
    results = []
    results.append(check_not_null(emp, "employee_id"))
    results.append(check_unique(emp, "employee_id"))
    results.append(check_email_simple(emp, "email"))
    results.append(check_date_past(emp, "hire_date"))
    results.append(check_term_after_hire(emp, "hire_date", "termination_date"))
    results.append(check_numeric_range(emp, "salary", 1000, 1000000))
    results.append(check_fk(emp, "department_id", dept, "department_id"))

    summary = {"run_id": datetime.now().isoformat(), "results": results}
    out_json = os.path.join(base, "reports_summary.json")
    with open(out_json, "w") as f:
        json.dump(summary, f, indent=2)
    print("Validation summary:")
    for r in results:
        details = {k:v for k,v in r.items() if k not in ['check','passed']}
        print(f"- {r['check']}: passed={r['passed']}, details={details}")
    print("\nWrote machine summary to", out_json)

if __name__ == '__main__':
    run(os.getcwd())

Validation summary:
- not_null:employee_id: passed=True, details={'null_count': 0}
- unique:employee_id: passed=True, details={'dup_count': 0}
- email_format:email: passed=True, details={'fail_count': 0}
- date_past:hire_date: passed=True, details={'future_count': 0, 'parse_errors': 0}
- term_after_hire: passed=True, details={'bad_count': 0}
- numeric_range:salary: passed=True, details={'below': 0, 'above': 0, 'non_numeric': 0}
- foreign_key:department_id: passed=True, details={'missing_count': 0}

Wrote machine summary to /content/reports_summary.json


## 3. View summary report  
The validation report below summarizes all checks and their results.  

In [11]:
import json
with open("reports_summary.json") as f:
    print(json.dumps(json.load(f), indent=2))

{
  "run_id": "2025-11-09T15:32:39.265582",
  "results": [
    {
      "check": "not_null:employee_id",
      "passed": true,
      "null_count": 0
    },
    {
      "check": "unique:employee_id",
      "passed": true,
      "dup_count": 0
    },
    {
      "check": "email_format:email",
      "passed": true,
      "fail_count": 0
    },
    {
      "check": "date_past:hire_date",
      "passed": true,
      "future_count": 0,
      "parse_errors": 0
    },
    {
      "check": "term_after_hire",
      "passed": true,
      "bad_count": 0
    },
    {
      "check": "numeric_range:salary",
      "passed": true,
      "below": 0,
      "above": 0,
      "non_numeric": 0
    },
    {
      "check": "foreign_key:department_id",
      "passed": true,
      "missing_count": 0
    }
  ]
}


## 4. Conclusion  
All data validation checks passed successfully.  
This project demonstrates how to generate synthetic HR datasets and apply automated validation pipelines for data quality assurance.  
The approach can be extended for payroll, attendance, or compliance datasets in HR analytics.